In [1]:
from importlib.machinery import SourceFileLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

In [12]:
%run GetPymatgenFeatures.py

StrToComposition:   0%|          | 0/1687 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1687 [00:00<?, ?it/s]

DensityFeatures:   0%|          | 0/1687 [00:00<?, ?it/s]

AtomicOrbitals:   0%|          | 0/1687 [00:00<?, ?it/s]

BandCenter:   0%|          | 0/1687 [00:00<?, ?it/s]

IonProperty:   0%|          | 0/1687 [00:00<?, ?it/s]

Stoichiometry:   0%|          | 0/1687 [00:00<?, ?it/s]

MultipleFeaturizer:   0%|          | 0/1687 [00:00<?, ?it/s]

AttributeError: 'list' object has no attribute 'to_pickle'

In [8]:
StructureFeatures

NameError: name 'StructureFeatures' is not defined

In [28]:
def get_plot_filename(plotname, minorcase):
    return 'graphs/{}_{}_{}_{}_{}_{}.pdf'.format(plotname, minorcase, CASE, MODEL, CUTOFF, criterion)

def get_table_filename(tablename, minorcase):
    return 'tables/{}_{}_{}_{}_{}_{}.csv'.format(tablename, minorcase, CASE, MODEL, CUTOFF, criterion)

def with_and_without_outliers(thisfeature: pd.core.series.Series, low = None, hig = None):
    spread = thisfeature.max() - thisfeature.min()
    if low == None:
        low = thisfeature.min()-spread*0.05
    if hig == None:
        hig = thisfeature.max()+spread*0.05
        
    fig, ax = plt.subplots(1,2,figsize=(30,10))
    h3=thisfeature.hist(bins=100, fill='k', ax=ax[1], density=True, label='data with outliers')
    h1=thisfeature[(thisfeature>low) & (thisfeature<hig)].hist(bins=100, ax=ax[0],density=True,label='current curated data (no outliers)')
    ax[0].legend()
    ax[0].set_xlabel('$E_f$ (eV)', x=1, labelpad=20)
    ax[0].set_ylabel('density counts', labelpad = 20)
    l = ax[1].legend()
    return fig


def compare_features(featureone, titleone, featuretwo, titletwo):
    fig, ax = plt.subplots(1,2,figsize=(30,10))
    h3=featureone.hist(bins=100, fill='k', ax=ax[1], density=True, label='total eneregy')
    h1=featuretwo.hist(bins=100, ax=ax[0],density=True,label='$\Delta E_f$')
    ax[1].set_xlabel(titleone, labelpad=20)
    ax[0].set_xlabel(titletwo, labelpad = 20)
    return fig
    #l = ax[1].legend()

def plot_pair(feature1, feature2, huefeature=None):
    if huefeature is None:
        huefeature = 'Class'
    fig, ax = plt.subplots()
    sns.scatterplot(data = data_w_classes, 
                    x=feature1,
                    y = feature2, ax = ax, hue=huefeature,
                    s=200)
    ax.set_xlabel(feature_titles[feature1])
    ax.set_ylabel(feature_titles[feature2])
    ax.legend(bbox_to_anchor=[1,1], markerscale=2)
    return fig

In [29]:
CASE = 'INITIAL'  #, 'INITIAL', RELAXED
MODEL = 'ATOMIC' #, 'ORTHOGONAL', 'CANONICAL'
CUTOFF = '' # 'TABLECUTOFF', 'HISTCUTOFF'
criterion = 'train_score'  # test_score, train_score, err

In [30]:
# from SourceDevelopementVersion import Featurizer, FeatureConcatenate
%run SourceDevelopementVersion.py

In [32]:
from BopFoxFeaturizer.Featurizer import Featurizer
from BopFoxFeaturizer.FeatureConcatenate import FeatureConcatenate

ModuleNotFoundError: No module named 'BopFoxFeaturizer.Featurizer'; 'BopFoxFeaturizer' is not a package

In [33]:
BS = pd.read_pickle('parsedbs.pkl')
Features = Featurizer(BS)
groundstates = Features.get_ground_states_energies()
BS['EF'] = Features.get_formation_energy(groundstates)

In [34]:
BS = BS[(BS['EF'] > -1) & (BS['EF']< 2)]
BS = BS[(BS['E0']>-100) & (BS['E0']<10)]
BS = BS[(BS['B0'])>0 & (BS['B0']<500)]
BS.dropna(how='any', axis=0,inplace=True)

In [35]:
AllFeatures = pd.concat([AtomicFeaturesMagpie, DensitiFeatures, CompositionFeatures,Features.MagFeature, BS[['nelem','num_atoms']]], axis = 1)

In [36]:
search_to_drop = AllFeatures.columns.str.contains('range|min|max')
columns_to_drop = AllFeatures.loc[:,search_to_drop].columns
AllFeatures.drop(columns=columns_to_drop, inplace=True)

In [12]:
AllFeatures

,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData mean MendeleevNumber,MagpieData avg_dev MendeleevNumber,MagpieData mode MendeleevNumber,MagpieData mean AtomicWeight,MagpieData avg_dev AtomicWeight,MagpieData mode AtomicWeight,MagpieData mean MeltingT,...,avg ionic char,0-norm,2-norm,3-norm,5-norm,7-norm,10-norm,MagConfig,nelem,num_atoms
index,,,,,,,,,,,,,,,,,,,,,
Co_pv6W_sv6.C14-BBA.FM,69.727273,7.768595,74.0,51.636364,1.157025,51.0,172.484836,20.645753,183.8400,3519.818182,...,0.004626,2,0.913625,0.909394,0.909093,0.909091,0.909091,2,2.0,12.0
Co_pv6W_sv6.C14-BBA.NM,69.727273,7.768595,74.0,51.636364,1.157025,51.0,172.484836,20.645753,183.8400,3519.818182,...,0.004626,2,0.913625,0.909394,0.909093,0.909091,0.909091,0,2.0,12.0
Cr_pv6W_sv2.D0_19-A3B.FM,62.461538,17.751479,74.0,50.538462,0.710059,51.0,153.414485,46.808485,183.8400,3345.384615,...,0.020466,2,0.803101,0.776092,0.769604,0.769255,0.769231,2,2.0,8.0
Cr_pv6W_sv2.D0_19-A3B.NM,62.461538,17.751479,74.0,50.538462,0.710059,51.0,153.414485,46.808485,183.8400,3345.384615,...,0.020466,2,0.803101,0.776092,0.769604,0.769255,0.769231,0,2.0,8.0
Cr_pv16Co_pv4W_sv10.sigma-CBAAC.FM,41.066667,21.955556,24.0,50.866667,1.991111,49.0,96.869013,57.980658,51.9961,2630.066667,...,0.023840,3,0.642910,0.576008,0.543235,0.536132,0.533816,2,3.0,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Cr_pv20Co_pv2W_sv8.sigma-BAACA.FM,37.533333,19.448889,24.0,50.133333,1.511111,49.0,87.616946,51.318962,51.9961,2556.533333,...,0.022026,3,0.721110,0.680809,0.668028,0.666823,0.666674,2,3.0,30.0
Cr_pv20Co_pv2W_sv8.sigma-BAACA.NM,37.533333,19.448889,24.0,50.133333,1.511111,49.0,87.616946,51.318962,51.9961,2556.533333,...,0.022026,3,0.721110,0.680809,0.668028,0.666823,0.666674,0,3.0,30.0
Co_pv13W_sv16.chi-ABAB.NM,70.468208,6.532794,74.0,51.526012,0.972969,51.0,174.453940,17.361499,183.8400,3550.196532,...,0.003890,2,0.927903,0.925021,0.924856,0.924855,0.924855,0,2.0,29.0


In [50]:
fig = px.histogram(AllFeatures, x='MagpieData mode NfValence')

In [13]:
AllFeatures.dropna(inplace=True)

In [14]:
#to_remove =  AllFeatures.columns.str.contains('Np|Nf|mode Number|mode Mendeleev|Row|mode Nd|mode')
#AllFeatures.drop(columns=AllFeatures.columns[to_remove],inplace=True)

In [20]:
bestfeats = {}
bestscores = {}
FC = {}

In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [19]:
X_train, X_test, Y_train, Y_test = train_test_split(AllFeatures, BS['EF'].loc[AllFeatures.index])

In [41]:
CASE = 'initial'
criterion = 'test_score'

In [42]:
del FeatureConcatenate
FeatureConcatenate = SourceFileLoader('FeatureConcate','/home/storage/fortimtb/CuadernoTrabajo/bopfoxfeaturizer/BopFoxFeaturizer/FeatureConcatenate.py'
                                     ).load_module().FeatureConcatenate

param_grid = {} 
Bestfeatsatomic = {} 
Bestscoresatomic = {}
FC['RandomForest_atomic'] = FeatureConcatenate(
    pd.concat([AllFeatures, BS['EF']], axis=1), #DATA, 
    RandomForestRegressor(),
    model_params=param_grid,
    feature_list=AllFeatures.columns,
    data_target='EF',
    criterion = criterion
#    sample_weights = w_train, #Classes['Weights']
)

Bestfeatsatomic['RandomForest_atomic'], Bestscoresatomic['RandomForest_atomic'] = FC['RandomForest_atomic'].build_features_list(
#    best_feature_proposal=['NSC_bn_1'],
    pass_force_refit=True,
    report_prefix='RandomForest_atomic_'+CASE,
)

procesing '[]' with 'MagpieData avg_dev Electronegativity' ... ::   0%|                          | 0/83 [00:00<?, ?it/s]

Refitting ..


procesing '[]' with 'MagConfig' ... :: 100%|############################################| 83/83 [01:06<00:00,  1.24it/s]


fitting has finished


procesing '['MagpieData avg_dev AtomicWeight']' with 'MagpieData mean GSvolume_pa' ... ::   0%|  | 0/62 [00:00<?, ?it/s]

...

fitting has finished


procesing '['MagpieData avg_dev AtomicWeight', 'num_atoms', 'MagpieData mode NfValence', '2-norm', 'compound possible', 

Refitting ..


procesing '['MagpieData avg_dev AtomicWeight', 'num_atoms', 'MagpieData mode NfValence', '2-norm', 'compound possible', 


fitting has finished


In [44]:
thisdata = pd.DataFrame(
    np.array([[x,y] for x,y in zip(Bestfeatsatomic['RandomForest_atomic'], Bestscoresatomic['RandomForest_atomic'])])
)

In [45]:
thisdata.to_csv(get_table_filename('AtomicFeatures',FC['RandomForest_atomic'].criterion))

In [46]:
thisdata

,0,1
0,MagpieData avg_dev AtomicWeight,0.11389812301154732
1,num_atoms,0.11065527605525928
2,MagpieData mode NfValence,0.10798038840849195
3,2-norm,0.10599211478467739
4,compound possible,0.10594837011673776
5,MagpieData mode GSbandgap,0.10524430357380156
6,HOMO_energy,0.10623842097180065
7,MagpieData mean NfUnfilled,0.10486451148988926
8,MagpieData avg_dev NpValence,0.105795016507788
9,MagpieData mode NpUnfilled,0.10607231440087368


In [ ]:
model = RandomForestRegressor()

In [ ]:
model.fit(BS[components], BS['EF'])

In [ ]:
Ypredict = model.predict(BS[components])

In [ ]:
mean_squared_error(BS['EF'], Ypredict)

In [ ]:
%store  Bestfeatsatomic

In [ ]:
%store Bestscoresatomic

In [ ]:
CompositionFeatures

In [ ]:
AllFeatures.columns

In [ ]:
to_remove =  AllFeatures.columns.str.contains('Np|Nf|mode Number|mode Mendeleev|Row|mode Nd|mode')

In [ ]:
AllFeatures.drop(columns=AllFeatures.columns[to_remove],inplace=True)

In [ ]:
AllFeatures.columns

In [ ]:
model = RandomForestRegressor()

In [ ]:
model.fit(,BS['EF'].loc[AllFeatures.index])

In [ ]:
YPredict = model.predict(AllFeatures)

In [ ]:
mean_squared_error(BS['EF'].loc[AllFeatures.index], YPredict, squared=False)